<a href="https://colab.research.google.com/github/AlmerindaEstudo/AlmerindaEstudo/blob/main/C%C3%B3pia_de_avanti_desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
COLAB - Passo 1: instalar e autenticar a API do Kaggle (token digitado na hora)
------------------------------------------------------------------------------
Ao rodar esta célula, vão aparecer dois campos para você digitar/colar:
- seu usuário do Kaggle (username, visível no seu perfil)
- sua chave (key) do arquivo kaggle.json
Nada fica gravado na célula/código - só fica em memória durante a sessão.

IMPORTANTE: rode este script (download do dataset) e, em seguida, o
split_data.py (corrigido, com SAMPLES_PER_CLASS=200) APENAS UMA VEZ.
O manifest.csv gerado deve ser reaproveitado por todos os notebooks
(1, 2, 3 e 4) para garantir que todos usem as mesmas imagens.
"""

!pip install -q kaggle

import os
import json
from getpass import getpass

KAGGLE_USERNAME = input("Digite seu usuário do Kaggle: ").strip()
KAGGLE_KEY = getpass("Cole sua chave (key) do Kaggle: ").strip()

os.makedirs("/root/.kaggle", exist_ok=True)
kaggle_credentials = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_credentials, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("\nAutenticação configurada com sucesso.")

# ============================================================
# COLAB - Passo 2: baixar o dataset direto do link do Kaggle
# ============================================================
# Link do dataset: https://www.kaggle.com/datasets/innominate817/hagrid-classification-512p
# O identificador usado pela API é: innominate817/hagrid-classification-512p
!kaggle datasets download -d innominate817/hagrid-classification-512p -p /content/dataset --unzip
print("\nDataset baixado e extraído em /content/dataset")

# ============================================================
# COLAB - Passo 3: inspecionar a estrutura de pastas
# ============================================================
for root, dirs, filenames in os.walk("/content/dataset"):
    print(root, "->", dirs[:5], f"({len(filenames)} arquivos)")
    if root.count("/") > 4:  # evita imprimir a árvore inteira
        break

# Depois desta célula: rode split_data.py (com SAMPLES_PER_CLASS=200)
# UMA ÚNICA VEZ para gerar o manifest.csv que será usado pelos 4 notebooks.

Digite seu usuário do Kaggle: Almerinda Gomes
Cole sua chave (key) do Kaggle: ··········

Autenticação configurada com sucesso.
Dataset URL: https://www.kaggle.com/datasets/innominate817/hagrid-classification-512p
License(s): CC-BY-SA-4.0
100% 11.7G/11.7G [02:25<00:00, 86.3MB/s]


Dataset baixado e extraído em /content/dataset
/content/dataset -> ['hagrid-classification-512p'] (0 arquivos)
/content/dataset/hagrid-classification-512p -> ['two_up_inverted', 'three', 'two_up', 'palm', 'stop_inverted'] (0 arquivos)
/content/dataset/hagrid-classification-512p/two_up_inverted -> [] (28077 arquivos)
/content/dataset/hagrid-classification-512p/three -> [] (27910 arquivos)
/content/dataset/hagrid-classification-512p/two_up -> [] (29564 arquivos)
/content/dataset/hagrid-classification-512p/palm -> [] (28317 arquivos)
/content/dataset/hagrid-classification-512p/stop_inverted -> [] (28753 arquivos)
/content/dataset/hagrid-classification-512p/ok -> [] (27900 arquivos)
/content/dataset/hagrid-classi

In [2]:
"""
split_data.py
--------------
Gera um manifest.csv (filepath, label, split) padronizado, usado pelos
4 notebooks para garantir as mesmas classes, a mesma quantidade de
imagens por classe (limitada a SAMPLES_PER_CLASS) e o mesmo split
treino/val/teste.

Rodar UMA ÚNICA VEZ antes de treinar qualquer um dos 4 modelos.
Depois disso, todos os notebooks (1, 2, 3, 4) devem ler o mesmo
manifest.csv gerado aqui — não rodar este script de novo entre eles,
senão o sorteio muda.
"""

import os
import random
from pathlib import Path
import pandas as pd

RANDOM_SEED = 42
CLASSES = ["call", "dislike", "fist", "four", "like", "mute", "ok", "one", "palm", "peace"]
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.8, 0.1, 0.1

# Teto de imagens por classe. Ajuste aqui (ex: 100 ou 200) conforme o
# tempo de treino que você quer ter. Se uma classe tiver menos imagens
# que isso, ela é limitada ao seu próprio total (e um aviso é impresso).
SAMPLES_PER_CLASS = 3000

# Candidatos de caminho: no Kaggle o dataset fica em /kaggle/input,
# no Colab (após baixar via API) fica em /content/dataset. O nome da
# pasta interna costuma vir duplicado (ex: .../hagrid-classification-512p/hagrid-classification-512p).
DATASET_DIR_NAME = "hagrid-classification-512p"
INPUT_ROOTS = ["/content/dataset", "/kaggle/input", "/content"]


def find_data_path(dataset_dir_name=DATASET_DIR_NAME, input_roots=INPUT_ROOTS, classes=CLASSES):
    """Procura, dentro das raízes conhecidas (Colab e Kaggle), a pasta que
    realmente contém as subpastas das classes (ex: .../call, .../dislike, ...),
    testando diferentes níveis de aninhamento."""
    existing_roots = [r for r in input_roots if os.path.exists(r)]
    if not existing_roots:
        raise FileNotFoundError(
            f"Nenhuma das raízes {input_roots} existe. Confirme se o dataset já "
            f"foi baixado (Colab) ou anexado ao notebook (Kaggle)."
        )

    for input_root in existing_roots:
        print(f"Verificando em: {input_root} ->", os.listdir(input_root))

        candidates = [
            Path(input_root) / dataset_dir_name,
            Path(input_root) / dataset_dir_name / dataset_dir_name,
            Path(input_root) / dataset_dir_name / dataset_dir_name / dataset_dir_name,
        ]

        for candidate in candidates:
            if candidate.exists():
                existing_classes = [c for c in classes if (candidate / c).exists()]
                print(f"  Testando: {candidate}  ->  {len(existing_classes)}/{len(classes)} classes encontradas")
                if len(existing_classes) == len(classes):
                    return str(candidate)

    # Última tentativa: busca recursiva por uma pasta que contenha
    # todas as 10 classes como subpastas diretas.
    print("Caminhos padrão não bateram. Fazendo busca recursiva (pode demorar um pouco)...")
    for input_root in existing_roots:
        for root, dirs, _ in os.walk(input_root):
            if all(c in dirs for c in classes):
                print(f"Encontrado via busca recursiva: {root}")
                return root

    raise FileNotFoundError(
        "Não foi possível localizar automaticamente a pasta com as 10 classes. "
        "Rode manualmente: import os; os.listdir('/content/dataset') (Colab) ou "
        "os.listdir('/kaggle/input') (Kaggle), explore as subpastas para achar "
        "o caminho certo, e passe-o em build_manifest(data_path='...')."
    )


def build_manifest(data_path=None, classes=CLASSES, seed=RANDOM_SEED,
                    samples_per_class=SAMPLES_PER_CLASS,
                    output_path="manifest.csv"):
    random.seed(seed)

    if data_path is None:
        data_path = find_data_path()
    print(f"\nUsando data_path: {data_path}\n")

    # Coleta os arquivos de cada classe
    class_files = {}
    for c in classes:
        class_dir = Path(data_path) / c
        if not class_dir.exists():
            raise FileNotFoundError(f"Pasta da classe '{c}' não encontrada em: {class_dir}")
        files = sorted(class_dir.glob("*.*"))
        class_files[c] = files
        print(f"  {c}: {len(files)} imagens disponíveis")

    if any(len(f) == 0 for f in class_files.values()):
        raise ValueError(
            "Pelo menos uma classe ficou com 0 imagens. Verifique se o data_path "
            "está correto e se as extensões dos arquivos são reconhecidas."
        )

    # Teto por classe: usa samples_per_class, mas nunca mais que o total
    # disponível na classe (evita duplicar imagens).
    min_available = min(len(f) for f in class_files.values())
    cap = min(samples_per_class, min_available)
    if cap < samples_per_class:
        print(
            f"\nAviso: samples_per_class={samples_per_class}, mas a menor classe "
            f"só tem {min_available} imagens. Usando cap={cap} para todas as classes."
        )
    print(f"\nUsando {cap} imagens por classe (fixo, mesma seed={seed} em toda execução).")

    records = []
    for c in classes:
        files = class_files[c][:]
        random.shuffle(files)          # shuffle determinístico (mesma seed)
        files = files[:cap]            # mesmo teto para todas as classes

        n = len(files)
        n_train = int(n * TRAIN_RATIO)
        n_val = int(n * VAL_RATIO)

        for i, f in enumerate(files):
            if i < n_train:
                split = "train"
            elif i < n_train + n_val:
                split = "val"
            else:
                split = "test"
            records.append({"filepath": str(f), "label": c, "split": split})

    df = pd.DataFrame(records)
    df.to_csv(output_path, index=False)

    print(f"\nManifest salvo em: {output_path}")
    print(df.groupby(["split", "label"]).size())
    return df


if __name__ == "__main__":
    build_manifest()

Verificando em: /content/dataset -> ['hagrid-classification-512p']
  Testando: /content/dataset/hagrid-classification-512p  ->  10/10 classes encontradas

Usando data_path: /content/dataset/hagrid-classification-512p

  call: 27984 imagens disponíveis
  dislike: 28345 imagens disponíveis
  fist: 27672 imagens disponíveis
  four: 28837 imagens disponíveis
  like: 27591 imagens disponíveis
  mute: 28724 imagens disponíveis
  ok: 27900 imagens disponíveis
  one: 28395 imagens disponíveis
  palm: 28317 imagens disponíveis
  peace: 28216 imagens disponíveis

Usando 3000 imagens por classe (fixo, mesma seed=42 em toda execução).

Manifest salvo em: manifest.csv
split  label  
test   call        300
       dislike     300
       fist        300
       four        300
       like        300
       mute        300
       ok          300
       one         300
       palm        300
       peace       300
train  call       2400
       dislike    2400
       fist       2400
       four       2400

In [3]:
"""
metrics_utils.py
-----------------
Funções compartilhadas para coletar métricas de desempenho e de custo
computacional de forma padronizada nos 4 notebooks.

Sem alterações de lógica em relação ao original: este arquivo não lida
com amostragem de imagens (isso é responsabilidade do split_data.py),
então continua igual e é seguro reaproveitar nos notebooks 1, 2, 3 e 4.
"""
import time
import json
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


def evaluate_classification(y_true, y_pred, labels):
    """Calcula acurácia, precision/recall/F1 (macro) e matriz de confusão."""
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    return {
        "accuracy": float(acc),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "confusion_matrix": cm.tolist(),
    }


def measure_inference_time(predict_fn, sample_input, n_runs=50, n_warmup=5):
    """Mede o tempo médio de inferência (segundos) para 1 amostra/lote."""
    for _ in range(n_warmup):
        predict_fn(sample_input)
    start = time.time()
    for _ in range(n_runs):
        predict_fn(sample_input)
    elapsed = time.time() - start
    return elapsed / n_runs


def save_report(report, path="report.json"):
    """Salva o relatório de métricas em JSON para consolidar no ranking final."""
    with open(path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"Relatório salvo em: {path}")

In [4]:

import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import ViTModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import json

# ============================================================
# CONFIGURAÇÃO
# ============================================================
SEED = 42
CLASSES = ["call", "dislike", "fist", "four", "like", "mute", "ok", "one", "palm", "peace"]
label2id = {c: i for i, c in enumerate(CLASSES)}
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

IMG_SIZE = 224
MODEL_CHECKPOINT = "google/vit-base-patch16-224-in21k"
BATCH_SIZE = 64

random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================
# 1. LOCALIZAR O DATASET E GERAR O MANIFEST (3000 imagens/classe)
# ============================================================
def find_data_path(dataset_dir_name="hagrid-classification-512p",
                    input_roots=("/content/dataset", "/kaggle/input", "/content"),
                    classes=CLASSES):
    for input_root in input_roots:
        if not os.path.exists(input_root):
            continue
        for depth in range(1, 4):
            candidate = Path(input_root).joinpath(*([dataset_dir_name] * depth))
            if candidate.exists():
                found = [c for c in classes if (candidate / c).exists()]
                if len(found) == len(classes):
                    return str(candidate)
    for input_root in input_roots:
        if not os.path.exists(input_root):
            continue
        for root, dirs, _ in os.walk(input_root):
            if all(c in dirs for c in classes):
                return root
    raise FileNotFoundError("Dataset não encontrado. Rode a célula de download do Kaggle antes desta.")


def build_manifest(manifest_path="manifest.csv", max_per_class=3000):
    if os.path.exists(manifest_path):
        print(f"Reaproveitando manifest existente: {manifest_path}")
        return pd.read_csv(manifest_path)

    data_path = find_data_path()
    class_files = {c: sorted(Path(data_path).joinpath(c).glob("*.*")) for c in CLASSES}
    min_count = min(len(f) for f in class_files.values())
    n_per_class = min(max_per_class, min_count)
    print(f"Imagens por classe usadas: {n_per_class}")

    records = []
    for c in CLASSES:
        files = class_files[c][:]
        random.shuffle(files)
        files = files[:n_per_class]
        n_train = int(len(files) * 0.8)
        n_val = int(len(files) * 0.1)
        for i, f in enumerate(files):
            split = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
            records.append({"filepath": str(f), "label": c, "split": split})

    df = pd.DataFrame(records)
    df.to_csv(manifest_path, index=False)
    return df


manifest = build_manifest(max_per_class=3000)


# ============================================================
# 2. EXTRAÇÃO DE EMBEDDINGS - passa cada imagem pelo ViT UMA VEZ
# ============================================================
tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])


class ImageOnlyDataset(Dataset):
    def __init__(self, split):
        self.data = manifest[manifest["split"] == split].reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        row = self.data.iloc[i]
        img = tf(Image.open(row["filepath"]).convert("RGB"))
        return img, label2id[row["label"]]


backbone = ViTModel.from_pretrained(MODEL_CHECKPOINT).to(device)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False


@torch.no_grad()
def extract_embeddings(split_name):
    loader = DataLoader(ImageOnlyDataset(split_name), batch_size=BATCH_SIZE, num_workers=2)
    all_embeddings, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        hidden = backbone(pixel_values=imgs).last_hidden_state
        cls_token = hidden[:, 0, :]           # resumo geral da imagem
        patch_mean = hidden[:, 1:, :].mean(1)  # média dos "pedacinhos" - detalhe espacial extra
        emb = torch.cat([cls_token, patch_mean], dim=1)  # 768 + 768 = 1536 números
        all_embeddings.append(emb.cpu())
        all_labels.append(labels)
    return torch.cat(all_embeddings), torch.cat(all_labels)


print("\nExtraindo embeddings (isso passa cada imagem pelo ViT 1 única vez)...")
start_extract = time.time()
X_train, y_train = extract_embeddings("train")
X_val, y_val = extract_embeddings("val")
X_test, y_test = extract_embeddings("test")
extract_time = time.time() - start_extract
print(f"Extração concluída em {extract_time:.1f}s")
print(f"Treino: {X_train.shape} | Val: {X_val.shape} | Teste: {X_test.shape}")


# ============================================================
# 3. TREINO - só a cabeça pequena, em cima dos embeddings já prontos
#
# ============================================================
classifier = nn.Sequential(
    nn.Linear(1536, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, len(CLASSES))
).to(device)

optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)


class FocalLoss(nn.Module):

    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(self, logits, target):
        ce_loss = self.ce(logits, target)
        p_t = torch.exp(-ce_loss)  # probabilidade que o modelo deu pro acerto
        focal_term = (1 - p_t) ** self.gamma  # quanto menor a chance de acerto, maior o peso
        return (focal_term * ce_loss).mean()


loss_fn = FocalLoss(gamma=2.0)

X_train, y_train = X_train.to(device), y_train.to(device)
X_val, y_val = X_val.to(device), y_val.to(device)

# Mini-batches, não o dataset inteiro de uma vez - é isso que garante
# vários ajustes de peso por época, em vez de só 1. Continua rápido
# porque os embeddings já estão prontos (768 números, não imagens).
train_emb_dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_emb_loader = torch.utils.data.DataLoader(
    train_emb_dataset, batch_size=32, shuffle=True
)

EPOCHS = 60
PATIENCE = 10
best_val_loss = float("inf")
best_state = None
no_improve = 0

start_train = time.time()
for epoch in range(EPOCHS):
    classifier.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    for xb, yb in train_emb_loader:
        optimizer.zero_grad()
        outputs = classifier(xb)
        loss = loss_fn(outputs, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * xb.size(0)
        epoch_correct += (outputs.argmax(1) == yb).sum().item()
        epoch_total += xb.size(0)
    scheduler.step()

    classifier.eval()
    with torch.no_grad():
        val_outputs = classifier(X_val)
        val_loss = loss_fn(val_outputs, y_val).item()
        val_acc = (val_outputs.argmax(1) == y_val).float().mean().item()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Época {epoch+1}/{EPOCHS} | treino acc={epoch_correct/epoch_total:.3f} | "
              f"val loss={val_loss:.4f} val acc={val_acc:.3f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in classifier.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping na época {epoch+1}")
            break

train_time = time.time() - start_train
classifier.load_state_dict(best_state)
print(f"\nTreino da cabeça concluído em {train_time:.1f}s")


# ============================================================
# 4. AVALIAÇÃO - no conjunto de teste
# ============================================================
classifier.eval()
with torch.no_grad():
    X_test_dev = X_test.to(device)
    y_pred = classifier(X_test_dev).argmax(1).cpu().numpy()
y_true = y_test.numpy()

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
precision_c, recall_c, f1_c, support_c = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(len(CLASSES))
)

print(f"\nAcurácia: {acc:.3f} | F1 macro: {f1:.3f}")
print(f"\n{'Classe':<16}{'Precision':>10}{'Recall':>10}{'F1':>10}{'N':>8}")
for i, c in enumerate(CLASSES):
    print(f"{c:<16}{precision_c[i]:>10.3f}{recall_c[i]:>10.3f}{f1_c[i]:>10.3f}{support_c[i]:>8}")

# Tempo de inferência (1 imagem, incluindo passar pelo ViT + cabeça)
sample = X_test_dev[:1]
for _ in range(5):
    with torch.no_grad():
        classifier(sample)
n_runs = 50
start = time.time()
for _ in range(n_runs):
    with torch.no_grad():
        classifier(sample)
infer_time = (time.time() - start) / n_runs

# ============================================================
# 5. RELATÓRIO FINAL
# ============================================================
n_trainable = sum(p.numel() for p in classifier.parameters())
n_total = sum(p.numel() for p in backbone.parameters()) + n_trainable

report = {
    "notebook": "notebook_5_rapido",
    "accuracy": float(acc),
    "precision_macro": float(precision),
    "recall_macro": float(recall),
    "f1_macro": float(f1),
    "confusion_matrix": cm.tolist(),
    "per_class_report": [
        {"classe": CLASSES[i], "precision": float(precision_c[i]),
         "recall": float(recall_c[i]), "f1": float(f1_c[i]), "n_amostras": int(support_c[i])}
        for i in range(len(CLASSES))
    ],
    "num_params_total": int(n_total),
    "num_params_trainable": int(n_trainable),
    "extraction_time_seconds": extract_time,
    "train_time_seconds": train_time,
    "total_time_seconds": extract_time + train_time,
    "avg_inference_time_seconds": infer_time,
}
with open("report_notebook5.json", "w") as f:
    json.dump(report, f, indent=2)

print(f"\nTempo total (extração + treino): {(extract_time + train_time)/60:.1f} minutos")
print("Relatório salvo em report_notebook5.json")
print(report)

Dispositivo: cuda
Reaproveitando manifest existente: manifest.csv


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]


Extraindo embeddings (isso passa cada imagem pelo ViT 1 única vez)...
Extração concluída em 399.2s
Treino: torch.Size([24000, 1536]) | Val: torch.Size([3000, 1536]) | Teste: torch.Size([3000, 1536])
Época 1/60 | treino acc=0.682 | val loss=0.3064 val acc=0.779
Época 5/60 | treino acc=0.867 | val loss=0.2492 val acc=0.820
Época 10/60 | treino acc=0.922 | val loss=0.2743 val acc=0.826
Época 15/60 | treino acc=0.946 | val loss=0.3023 val acc=0.828
Early stopping na época 15

Treino da cabeça concluído em 25.0s

Acurácia: 0.844 | F1 macro: 0.843

Classe           Precision    Recall        F1       N
call                 0.930     0.887     0.908     300
dislike              0.916     0.950     0.933     300
fist                 0.910     0.913     0.912     300
four                 0.699     0.643     0.670     300
like                 0.908     0.890     0.899     300
mute                 0.955     0.997     0.976     300
ok                   0.782     0.767     0.774     300
one       